In [1]:
!pip install transformers

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 0. GPU 장치 및 기본 세팅
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 현재 사용할 장치: {device} (앙상블 엔진 가동 준비 완료!)")

# 데이터 불러오기
train_df = pd.read_csv('final_100k_train_set.csv')
test_df = pd.read_csv('test_set_clean_200.csv')
texts_train, y_train = train_df['text'].values, train_df['is_spoiler'].values
texts_test, y_test = test_df['text'].values, test_df['is_spoiler'].values

# 클래스 불균형 가중치 계산 (약 9.3배)
imbalance_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# =====================================================================
# [모델 1] 로지스틱 회귀 (단어 매칭 전문가) - 초고속 학습
# =====================================================================
print("\n[1/3] 🟢 머신러닝(Logistic Regression) 학습 중...")
tfidf_vec = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vec.fit_transform(texts_train)
X_test_tfidf = tfidf_vec.transform(texts_test)

lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# 테스트셋 예측 확률 뽑아두기
probs_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]
print("✅ 1번 모델 학습 및 확률 추출 완료!")

# =====================================================================
# [모델 2] 1D-CNN (구조적 패턴 전문가) - 고속 학습
# =====================================================================
print("\n[2/3] 🔵 딥러닝(1D-CNN) 학습 중...")
count_vec = CountVectorizer(max_features=15000, analyzer='word', ngram_range=(1, 2))
X_train_cnn = count_vec.fit_transform(texts_train).toarray()
X_test_cnn = count_vec.transform(texts_test).toarray()

class CNN_Dataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

cnn_train_loader = DataLoader(CNN_Dataset(X_train_cnn, y_train), batch_size=256, shuffle=True)
cnn_test_loader = DataLoader(CNN_Dataset(X_test_cnn, y_test), batch_size=256, shuffle=False)

class SpoilerCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(64 * 7500, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu, self.dropout = nn.ReLU(), nn.Dropout(0.5)
    def forward(self, x):
        x = self.relu(self.conv1(x.unsqueeze(1)))
        x = self.dropout(self.relu(self.fc1(self.pool(x).view(x.size(0), -1))))
        return self.fc2(x) # BCEWithLogitsLoss를 쓰기 위해 Sigmoid 제외

cnn_model = SpoilerCNN().to(device)
criterion_cnn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([imbalance_weight]).to(device))
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=0.001)

cnn_model.train()
for epoch in range(3): # 빠른 학습을 위해 3 Epoch
    for inputs, labels in cnn_train_loader:
        optimizer_cnn.zero_grad()
        loss = criterion_cnn(cnn_model(inputs.to(device)), labels.to(device))
        loss.backward()
        optimizer_cnn.step()

# 테스트셋 예측 확률 뽑아두기 (Sigmoid 적용하여 0~1 사이 확률로 변환)
cnn_model.eval()
probs_cnn = []
with torch.no_grad():
    for inputs, _ in cnn_test_loader:
        outputs = torch.sigmoid(cnn_model(inputs.to(device)))
        probs_cnn.extend(outputs.cpu().numpy().flatten())
probs_cnn = np.array(probs_cnn)
print("✅ 2번 모델 학습 및 확률 추출 완료!")

# =====================================================================
# [모델 3] KcELECTRA (문맥 이해 최강자) - 정밀 학습 (시간 소요)
# =====================================================================
print("\n[3/3] 🟣 대형 딥러닝(KcELECTRA) 학습 중... (시간이 제법 걸립니다. 커피 한잔!)")
tokenizer = AutoTokenizer.from_pretrained("beomi/KcELECTRA-base-v2022")
electra_model = AutoModelForSequenceClassification.from_pretrained("beomi/KcELECTRA-base-v2022", num_labels=1).to(device)

class ElectraDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return {key: val[idx] for key, val in self.encodings.items()}, self.labels[idx]

electra_train_loader = DataLoader(ElectraDataset(texts_train, y_train), batch_size=32, shuffle=True)
electra_test_loader = DataLoader(ElectraDataset(texts_test, y_test), batch_size=32, shuffle=False)

criterion_electra = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([imbalance_weight]).to(device))
optimizer_electra = optim.AdamW(electra_model.parameters(), lr=2e-5)

electra_model.train()
for epoch in range(2): # 대형 모델이라 2 Epoch만 해도 충분합니다.
    epoch_loss = 0
    for batch, labels in tqdm(electra_train_loader, desc=f"Electra Epoch {epoch+1}/2"):
        optimizer_electra.zero_grad()
        outputs = electra_model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        loss = criterion_electra(outputs.logits, labels.to(device))
        loss.backward()
        optimizer_electra.step()

# 테스트셋 예측 확률 뽑아두기
electra_model.eval()
probs_electra = []
with torch.no_grad():
    for batch, _ in electra_test_loader:
        outputs = electra_model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        probs = torch.sigmoid(outputs.logits).cpu().numpy().flatten()
        probs_electra.extend(probs)
probs_electra = np.array(probs_electra)
print("✅ 3번 모델 학습 및 확률 추출 완료!")

# =====================================================================
# 🏆 앙상블 (Ensemble) 결합 및 최종 평가 (소프트 보팅)
# =====================================================================
print("\n🔥 3대장 앙상블 소프트 보팅(Soft Voting) 진행 중...")
# 세 모델의 확률을 평균냅니다.
ensemble_probs = (probs_lr + probs_cnn + probs_electra) / 3.0

# 평균 확률이 50% 이상이면 스포일러(1)로 판정
ensemble_preds = (ensemble_probs >= 0.35).astype(int)

print("\n================ 🏆 앙상블 최종 성적표 🏆 ================")
print(classification_report(y_test, ensemble_preds, target_names=['정상(0)', '스포일러(1)']))
final_f1 = f1_score(y_test, ensemble_preds)
final_recall = recall_score(y_test, ensemble_preds)
print(f"🎯 최종 앙상블 F1-Score: {final_f1:.4f}")
print(f"📢 최종 앙상블 재현율(Recall): {final_recall:.4f}")
print("=========================================================")

# 5단계 Streamlit 배포를 위한 파일 저장 (모델 저장 시 용량 주의)
import joblib
joblib.dump(lr_model, 'ensemble_lr_model.pkl')
joblib.dump(tfidf_vec, 'ensemble_tfidf_vec.pkl')
torch.save(cnn_model.state_dict(), 'ensemble_cnn_model.pth')
joblib.dump(count_vec, 'ensemble_count_vec.pkl')
electra_model.save_pretrained('./ensemble_electra_model')
tokenizer.save_pretrained('./ensemble_electra_model')

print("\n✅ 모든 앙상블 자산이 안전하게 저장되었습니다! 이제 Streamlit 배포만 남았습니다!")

c:\Users\user\AppData\Local\anaconda3\envs\textmine26\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 현재 사용할 장치: cuda (앙상블 엔진 가동 준비 완료!)

[1/3] 🟢 머신러닝(Logistic Regression) 학습 중...
✅ 1번 모델 학습 및 확률 추출 완료!

[2/3] 🔵 딥러닝(1D-CNN) 학습 중...
✅ 2번 모델 학습 및 확률 추출 완료!

[3/3] 🟣 대형 딥러닝(KcELECTRA) 학습 중... (시간이 제법 걸립니다. 커피 한잔!)


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Electra Epoch 2/2: 100%|██████████| 3125/3125 [21:05<00:00,  2.47it/s]


✅ 3번 모델 학습 및 확률 추출 완료!

🔥 3대장 앙상블 소프트 보팅(Soft Voting) 진행 중...

================ 🏆 앙상블 최종 성적표 🏆 ================
              precision    recall  f1-score   support

       정상(0)       0.99      0.89      0.94       176
     스포일러(1)       0.54      0.92      0.68        24

    accuracy                           0.90       200
   macro avg       0.76      0.90      0.81       200
weighted avg       0.93      0.90      0.91       200

🎯 최종 앙상블 F1-Score: 0.6769
📢 최종 앙상블 재현율(Recall): 0.9167

✅ 모든 앙상블 자산이 안전하게 저장되었습니다! 이제 Streamlit 배포만 남았습니다!


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score, recall_score
import joblib
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 1D-CNN 모델 뼈대 (불러오기를 위해 반드시 필요)
class SpoilerCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(64 * 7500, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu, self.dropout = nn.ReLU(), nn.Dropout(0.5)
    def forward(self, x):
        x = self.relu(self.conv1(x.unsqueeze(1)))
        x = self.dropout(self.relu(self.fc1(self.pool(x).view(x.size(0), -1))))
        return self.fc2(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("🔄 인공지능 모델들을 깨우는 중...")

# 2. 방금 직접 정성껏 수정한 진짜 시험지 로드
test_df = pd.read_csv('test_set_clean_200.csv')
texts_test = test_df['text'].values
y_test = test_df['is_spoiler'].values

# 3. 모델 및 변환기 불러오기
# [ML 모델]
lr_model = joblib.load('ensemble_lr_model.pkl')
tfidf_vec = joblib.load('ensemble_tfidf_vec.pkl')

# [CNN 모델]
cnn_model = SpoilerCNN().to(device)
cnn_model.load_state_dict(torch.load('ensemble_cnn_model.pth', map_location=device))
cnn_model.eval()
count_vec = joblib.load('ensemble_count_vec.pkl')

# [ELECTRA 모델]
tokenizer = AutoTokenizer.from_pretrained('./ensemble_electra_model')
electra_model = AutoModelForSequenceClassification.from_pretrained('./ensemble_electra_model').to(device)
electra_model.eval()

print("📝 새로 수정한 시험지로 채점을 시작합니다!")

# 4. 각 모델별 예측 수행
# ML 예측
X_test_tfidf = tfidf_vec.transform(texts_test)
probs_lr = lr_model.predict_proba(X_test_tfidf)[:, 1]

# CNN 예측
X_test_cnn = count_vec.transform(texts_test).toarray()
X_test_cnn_tensor = torch.tensor(X_test_cnn, dtype=torch.float32).to(device)
with torch.no_grad():
    probs_cnn = torch.sigmoid(cnn_model(X_test_cnn_tensor)).cpu().numpy().flatten()

# ELECTRA 예측 (배치 처리 불필요 시 한 번에 진행, 200건이므로 가능)
encodings = tokenizer(texts_test.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
with torch.no_grad():
    outputs = electra_model(input_ids=encodings['input_ids'].to(device), attention_mask=encodings['attention_mask'].to(device))
    probs_electra = torch.sigmoid(outputs.logits).cpu().numpy().flatten()

# 5. 앙상블 투표 및 찐 성적표 도출
ensemble_probs = (probs_lr + probs_cnn + probs_electra) / 3.0
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

print("\n================ 🏆 찐(Real) 앙상블 최종 성적표 🏆 ================")
print(classification_report(y_test, ensemble_preds, target_names=['정상(0)', '스포일러(1)']))
final_f1 = f1_score(y_test, ensemble_preds)
final_recall = recall_score(y_test, ensemble_preds)
print(f"🎯 찐 최종 F1-Score: {final_f1:.4f}")
print(f"📢 찐 최종 재현율(Recall): {final_recall:.4f}")
print("================================================================")